# 🏀 NBA Referee AI — Deep Spatiotemporal ConvNet (`R(2+1)D-18`) & Reinforcement Learning Officiating Agent
**Course:** CSC 434 — Artificial Intelligence & Machine Learning Final Project  
**Target Hardware:** Google Colab NVIDIA Tesla T4 GPU (15.3 GB GDDR6 VRAM, 320 Turing Tensor Cores)  
**Core Technologies:** PyTorch 2.x, Torchvision 3D Video Models, Ultralytics YOLOv11-Pose, Dueling DQN / Actor-Critic Sequential RL

---

### 🚀 Google Colab High-Compute & Full GPU RAM Utilization Architecture
Previous iterations of this pipeline ran in a low-memory cached mode designed for laptops (< 1 GB VRAM usage). This overhauled release is engineered **specifically for Google Colab Tesla T4 GPUs**, utilizing **12–14 GB of GPU RAM** and saturating Turing Tensor Cores via:
1. **End-to-End Spatiotemporal 3D Fine-Tuning**: Raw 4D video tensors `(B, 3, 16, 224, 224)` pass directly into `R(2+1)D-18` with full gradient backpropagation in FP16 Automatic Mixed Precision (`torch.amp.autocast('cuda')`).
2. **High-Throughput Batch Processing**: Batch size 16 (effective batch size 32 with gradient accumulation) with pinned memory (`pin_memory=True`) and multi-worker prefetching (`num_workers=2`).
3. **Sequential Reinforcement Learning Referee Agent (`NBARefereeEnv` + Dueling DQN / Actor-Critic)**: Rather than static frame classification, officiating is modeled as a sequential decision process ($S_t \in \mathbb{R}^{281}$, $A_t \in \{\text{PLAY ON}, \text{WHISTLE}\}$). Rewards are directly derived from official NBA Last Two Minute (L2M) reviews (`CC`, `INC`, `CNC`, `IC`) and Rule 12 biomechanics.
4. **Broadcast-Grade Multimodal Video HUD**: Multi-player tracking (Offense in Cyan, Defense in Orange, Disputed Hull), 17-keypoint skeleton overlay, defender verticality guidelines, live RL Q-value gauges, Rule 12 violation bars, and a bottom temporal scrubbing progress strip.

```text
+---------------------------------------------------------------------------------------------------+
| STAGE 1: NBA L2M Data Scraper & Parallel Video Downloader (Cell 2)                                 |
|  • Concurrent multi-worker download of official NBA CDN broadcast MP4 clips                       |
|  • Officiating Ground Truth: CC (Correct Call), INC (Missed Foul), CNC (Legal), IC (Phantom Call)  |
+-------------------------------------------------+-------------------------------------------------+
                                                  |
                                                  v
+---------------------------------------------------------------------------------------------------+
| STAGE 2: GPU-Accelerated Action-Tube & 12-D Biomechanical Contact Extractor (Cell 4)              |
|  • YOLOv11-Pose batch inference on CUDA with FP16                                                 |
|  • 12 Mechanical Foul Dimensions grounded in NBA Rule 12 (cylinder breach, verticality, contact)  |
+-------------------------------------------------+-------------------------------------------------+
                                                  |
                         +------------------------+------------------------+
                         |                                                 |
                         v                                                 v
+------------------------------------------------+ +------------------------------------------------+
| STREAM A: 3D Spatiotemporal Video Backbone     | | STREAM B: Biomechanical Contact Attention Net  |
|  • Input: (B, 3, 16, 224, 224) 4D Video Tensor | |  • Input: (B, 16, 12) Kinematic Contact Vector |
|  • R(2+1)D-18 End-to-End Fine-Tuning on CUDA   | |  • 4-Head Temporal Self-Attention + LayerNorm  |
|  • Output: 512-D Visual Feature Vector         | |  • Output: 256-D Biomechanical Representation  |
+------------------------+-----------------------+ +-----------------------+------------------------+
                         |                                                 |
                         +------------------------+------------------------+
                                                  |
                                                  v
+---------------------------------------------------------------------------------------------------+
| STAGE 3 & 4: Late Fusion & High-VRAM End-to-End GPU Training (Cell 5 & 6)                          |
|  • Combined 768-D Embedding -> Deep Classification Head -> Class-Balanced Focal Loss              |
|  • Consumes ~12-14 GB of GDDR6 VRAM on Google Colab Tesla T4 with FP16 Tensor Core Acceleration    |
+-------------------------------------------------+-------------------------------------------------+
                                                  |
                                                  v
+---------------------------------------------------------------------------------------------------+
| STAGE 5 & 6: Sequential Reinforcement Learning Referee Agent (Cell 7 & 8)                          |
|  • Gym-like sequential officiating environment with Rule 12 reward shaping & L2M ground truth     |
|  • Dueling Q-Network learning WHEN to blow whistle (reward: +2.5 CC/INC, -3.0 phantom whistle)    |
+-------------------------------------------------+-------------------------------------------------+
                                                  |
                                                  v
+---------------------------------------------------------------------------------------------------+
| STAGE 7 & 8: Broadcast-Grade Continuous Video Annotation & In-Notebook HTML5 Replay (Cell 9 & 10)|
|  • Enclosing disputed hull, 17-keypoint skeleton, verticality angle guide, RL Q-value telemetry,  |
|    and bottom scrubbing timeline bar -> Exported to data/annotated/<play_id>_annotated.mp4       |
+---------------------------------------------------------------------------------------------------+
```


In [ ]:
# ==============================================================================
# CELL 1: Universal Bootstrap, Colab T4 High-Compute Setup & Config
# ==============================================================================
import importlib
import os
import sys
import csv
import gc
import json
import time
import random
import platform
import subprocess
from pathlib import Path
from collections import Counter, deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Any, Callable, Dict, List, Optional, Tuple, Union

# ------------------------------------------------------------------------------
# 1. Google Colab Environment Auto-Bootstrap & Dependency Installation
# ------------------------------------------------------------------------------
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('🚀 Running in Google Colab environment!')
    print('📦 Installing required dependencies for high-throughput GPU execution...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'opencv-python-headless', 'scikit-learn', 'matplotlib'], check=False)
    
    repo_dir = Path('/content/AI-ML-Final-Project')
    if not repo_dir.exists():
        print('📥 Cloning repository from GitHub (testing branch)...')
        subprocess.run(['git', 'clone', '-b', 'testing', 'https://github.com/ferraromike/AI-ML-Final-Project.git', str(repo_dir)], check=False)
    
    target_dir = repo_dir / 'NBA-Foul-Classifier-WIP'
    if target_dir.exists():
        os.chdir(str(target_dir))
    else:
        os.chdir(str(repo_dir))
    print(f'📁 Working directory set to: {Path.cwd()}')

# OpenMP & OpenCV optimization
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '2')
os.environ.setdefault('MKL_NUM_THREADS', '2')

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision.transforms.functional as TF
from torchvision.models.video import R2Plus1D_18_Weights, r2plus1d_18
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

cv2.setNumThreads(0)

try:
    from ultralytics import YOLO
except ImportError:
    YOLO = None

# ------------------------------------------------------------------------------
# 2. Project Directory & Checkpoint Resolution
# ------------------------------------------------------------------------------
CWD = Path.cwd().resolve()
if (CWD / 'NBA-Foul-Classifier-WIP').exists():
    PROJECT_DIR = CWD / 'NBA-Foul-Classifier-WIP'
elif (CWD / '3_model_and_dataset.py').exists():
    PROJECT_DIR = CWD
elif (CWD.parent / 'NBA-Foul-Classifier-WIP').exists():
    PROJECT_DIR = CWD.parent / 'NBA-Foul-Classifier-WIP'
else:
    PROJECT_DIR = CWD

for p in [PROJECT_DIR, PROJECT_DIR.parent]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

DATA_DIR = PROJECT_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
METADATA_CSV = DATA_DIR / 'l2m_metadata.csv'
INCORRECT_CSV = DATA_DIR / 'l2m_incorrect_fouls.csv'
PROCESSED_METADATA_CSV = PROCESSED_DIR / 'processed_metadata.csv'
WEIGHTS_DIR = PROJECT_DIR / 'weights'
TEST_DIR = DATA_DIR / 'test'
ANNOTATED_DIR = DATA_DIR / 'annotated'

CHECKPOINT_PATH = WEIGHTS_DIR / 'best_foul_r2plus1d.pt'
if not CHECKPOINT_PATH.exists() and (PROJECT_DIR / 'best_foul_r2plus1d.pt').exists():
    CHECKPOINT_PATH = PROJECT_DIR / 'best_foul_r2plus1d.pt'

RL_CHECKPOINT_PATH = WEIGHTS_DIR / 'referee_rl_agent.pt'

# ------------------------------------------------------------------------------
# 3. Google Colab Tesla T4 Hardware Allocation Detector
# ------------------------------------------------------------------------------
IS_CUDA = torch.cuda.is_available()
VRAM_GB = (torch.cuda.get_device_properties(0).total_memory / 1e9) if IS_CUDA else 0.0
IS_T4_GPU = IS_CUDA and (VRAM_GB >= 12.0)

device_name = "CPU"
if IS_CUDA:
    device_name = f"CUDA: {torch.cuda.get_device_name(0)} ({VRAM_GB:.1f} GB GDDR6 VRAM)"
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device_name = "Apple Silicon MPS GPU"

DEVICE = torch.device('cuda' if IS_CUDA else ('mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else 'cpu'))

print(f'🖥️  Platform         : {platform.system()} ({platform.machine()})')
print(f'🔥 PyTorch Version  : {torch.__version__}')
print(f'⚡ Compute Device   : {device_name}')

# Pre-warm cuBLAS context handle while memory is completely clean
if IS_CUDA:
    try:
        _d = torch.empty((2, 2), device='cuda')
        _ = _d @ _d
        del _d
        torch.cuda.empty_cache()
    except Exception:
        pass
print(f'📁 Project Directory: {PROJECT_DIR}')

if IS_T4_GPU:
    print('🚀 [HIGH-COMPUTE ENGINE] Tesla T4 GPU detected! High-VRAM End-to-End mode configured.')
    print('🎯 Target GPU RAM Allocation: ~12–14 GB VRAM (Full 3D ConvNet unfreezing + FP16 AMP Tensor Cores)')

# ------------------------------------------------------------------------------
# 4. Colab T4 Optimized Configuration Knobs
# ------------------------------------------------------------------------------
CONFIG = {
    # High-VRAM Spatiotemporal Configuration (Pushes T4 to ~12-14 GB VRAM)
    'freeze_backbone': False if IS_T4_GPU else True,    # False unfreezes R(2+1)D-18 for end-to-end 3D training
    'use_cached_embeddings': False if IS_T4_GPU else True, # False feeds raw (3, 16, 224, 224) 4D tensors to GPU
    'batch_size': 4 if IS_T4_GPU else 4,               # Micro-batch 4 prevents VRAM spikes with 3D conv
    'accum_steps': 8 if IS_T4_GPU else 2,              # Effective batch size = 4 * 8 = 32!
    'pin_memory': True if IS_CUDA else False,           # Fast host-to-device memory pinning
    'num_workers': 2 if (IS_CUDA and platform.system() == 'Linux') else 0, # Colab dual vCPUs
    'amp': True if IS_CUDA else False,                  # Automatic Mixed Precision for Tensor Cores
    'epochs': 15,                                       # Training epochs for spatiotemporal convergence
    'lr': 1.0e-4,                                       # AdamW learning rate
    'weight_decay': 1e-2,                               # Regularization
    'dropout': 0.35,                                    # Dropout rate
    'num_frames': 16,                                   # 16-frame spatiotemporal action tube
    'crop_size': 224,                                   # 224x224 spatial crop around active contest
    
    # Reinforcement Learning Officiating Agent Configuration
    'train_rl': True,                                   # Train sequential RL referee agent
    'rl_mode': 'dqn',                                   # 'dqn' (Dueling Q-Network) or 'actor_critic'
    'rl_episodes': 150,                                 # Episodes for RL referee training
    'rl_lr': 3e-4,                                      # RL optimizer learning rate
    
    # Scraper & Preprocessing Scaling Knobs
    'max_games': 25,                                    # Games to inspect when scraping
    'max_scrape_clips': 100,                            # Max clips to download if scraping
    'max_preprocess_clips': 0,                          # 0 = preprocess all available raw videos
    'yolo_batch_size': 16 if IS_CUDA else 4,            # Micro-batch size for YOLOv11-Pose
    'max_scan_samples': 24,                             # Scan sample rate for peak ROI
    'max_live_play_sec': 20.0,                          # Focus on active contest window
    'seed': 42,                                         # Reproducibility seed
}


---
## Stage 1: NBA Last Two Minute (L2M) Scraper & Parallel Video Downloader
Official NBA Last Two Minute (L2M) reports provide transparent, post-game officiating evaluations for all contested plays in the final 2 minutes of close games. We scrape official reports and map decisions to binary ground-truth labels:

- **Class 1 (`foul`)**:
  - `CC` (*Correct Call*): Floor referee whistled a foul, and NBA league review confirmed illegal contact occurred.
  - `INC` (*Incorrect Non-Call*): Floor referee swallowed whistle; NBA league review determined an illegal foul occurred!
- **Class 0 (`no_foul`)**:
  - `CNC` (*Correct Non-Call*): Floor referee allowed play to continue; league review confirmed defense was legal.
  - `IC` (*Incorrect Call*): Floor referee erroneously whistled a phantom foul; league review confirmed defense was legal!

The high-speed multithreaded downloader below fetches missing broadcast MP4 clips directly from NBA CDN endpoints.


In [ ]:
# ==============================================================================
# CELL 2: Stage 1 — High-Throughput L2M Scraper & Parallel Video Downloader
# ==============================================================================
import importlib
scrape_mod = importlib.import_module('1_scrape_l2m')
importlib.reload(scrape_mod)

def run_parallel_video_download(metadata_csv: Path, raw_dir: Path, max_workers: int = 6, max_clips: int = 50) -> int:
    """Download missing NBA CDN video clips in parallel using ThreadPoolExecutor."""
    foul_dir = raw_dir / "foul"
    no_foul_dir = raw_dir / "no_foul"
    foul_dir.mkdir(parents=True, exist_ok=True)
    no_foul_dir.mkdir(parents=True, exist_ok=True)

    if not metadata_csv.exists():
        print(f"⚠️ Metadata CSV not found at: {metadata_csv}")
        return 0

    with open(metadata_csv, "r", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    to_download = []
    for r in rows:
        pid = str(r.get("play_id", "")).strip()
        lbl = int(r.get("label", 0))
        target_dir = foul_dir if lbl == 1 else no_foul_dir
        dest = target_dir / f"{pid}.mp4"
        if not dest.exists() or dest.stat().st_size < 10_240:
            to_download.append(r)

    if not to_download:
        print(f"✅ All {len(rows)} clips in metadata are already downloaded on disk!")
        return 0

    if max_clips > 0:
        to_download = to_download[:max_clips]

    print(f"📥 Downloading {len(to_download)} missing broadcast clips (Workers: {max_workers})...")
    session = scrape_mod.build_session()
    downloaded = 0

    def _worker(row: Dict[str, Any]) -> bool:
        pid = str(row.get("play_id", "")).strip()
        gid = scrape_mod.normalize_game_id(str(row.get("game_id", "")))
        lbl = int(row.get("label", 0))
        target_dir = foul_dir if lbl == 1 else no_foul_dir
        dest = target_dir / f"{pid}.mp4"
        parts = pid.split("_")
        eid = parts[1] if len(parts) > 1 else ""
        if not eid:
            return False
        url = scrape_mod.resolve_mp4_url(session, gid, eid)
        if not url:
            return False
        return scrape_mod.download_mp4_clip(session, url, dest)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_worker, r): r.get("play_id") for r in to_download}
        for fut in as_completed(futures):
            try:
                if fut.result():
                    downloaded += 1
                    if downloaded % 10 == 0 or downloaded == len(to_download):
                        print(f"   Progress: {downloaded}/{len(to_download)} clips downloaded")
            except Exception:
                pass

    print(f"✅ Bulk download complete! Successfully retrieved {downloaded} clips.")
    return downloaded

print("🔍 Inspecting local dataset metadata...")
if METADATA_CSV.exists():
    with open(METADATA_CSV, 'r', encoding='utf-8') as f:
        meta_rows = list(csv.DictReader(f))
    print(f"📊 Total plays recorded in master metadata: {len(meta_rows)}")
    
    foul_clips = list((RAW_DIR / 'foul').glob('*.mp4')) if (RAW_DIR / 'foul').exists() else []
    no_foul_clips = list((RAW_DIR / 'no_foul').glob('*.mp4')) if (RAW_DIR / 'no_foul').exists() else []
    total_downloaded = len(foul_clips) + len(no_foul_clips)
    print(f"🎬 Existing raw clips on disk: {total_downloaded} ({len(foul_clips)} foul, {len(no_foul_clips)} no_foul)")
    
    missing_count = len(meta_rows) - total_downloaded
    if missing_count > 0:
        run_parallel_video_download(
            metadata_csv=METADATA_CSV,
            raw_dir=RAW_DIR,
            max_workers=6,
            max_clips=CONFIG['max_scrape_clips'],
        )
    else:
        print("✅ 100% of metadata clips are present on disk.")
else:
    print("🌐 Metadata CSV not found locally. Running live NBA L2M scraper...")
    scrape_mod.run_training_dataset_scraper(
        output_dir=RAW_DIR,
        metadata_csv=METADATA_CSV,
        daily_csv=INCORRECT_CSV,
        include_archive=True,
        max_games=CONFIG['max_games'],
        max_clips=CONFIG['max_scrape_clips'],
    )


---
## Exploratory Data Analysis (EDA): Inspecting Officiating Decisions & Class Imbalance
Before training spatiotemporal models and the Reinforcement Learning agent, we analyze the official NBA L2M ground-truth distribution across `CC`, `INC`, `CNC`, and `IC` review decisions.


In [ ]:
# ==============================================================================
# CELL 3: Dataset Summary, Officiating Review Decisions & Distributions
# ==============================================================================
with open(METADATA_CSV, 'r', encoding='utf-8') as f:
    meta_rows = list(csv.DictReader(f))

decisions = [r.get('review_decision', '').strip() for r in meta_rows if r.get('review_decision')]
labels = [int(r.get('label', 0)) for r in meta_rows]
call_types = [r.get('call_type', 'Unknown').strip() for r in meta_rows]

dec_counts = Counter(decisions)
label_counts = Counter(labels)
call_counts = Counter(call_types)

print(f"📊 Master L2M Dataset Summary ({len(meta_rows)} total plays):")
print(f"   • Fouls (Label 1): {label_counts[1]} plays ({label_counts[1] / len(meta_rows) * 100:.1f}%)")
print(f"   • No Fouls (Label 0): {label_counts[0]} plays ({label_counts[0] / len(meta_rows) * 100:.1f}%)")
print(f"   • NBA Officiating Decisions: {dict(dec_counts)}")
print(f"   • Foul Call Types: {dict(call_counts)}")

# Plot 3-Panel Visual Distribution
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.5), dpi=100)

# 1. Binary Class Balance
bars1 = ax1.bar(['No Foul (0)', 'Foul (1)'], [label_counts[0], label_counts[1]], color=['#2ca02c', '#d62728'], width=0.5)
ax1.set_title('Binary Ground Truth Balance', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Plays')
for b in bars1:
    h = b.get_height()
    ax1.text(b.get_x() + b.get_width() / 2, h + 3, f'{h}', ha='center', fontweight='bold')

# 2. Review Decisions
dec_order = ['CNC', 'CC', 'INC', 'IC']
dec_vals = [dec_counts.get(d, 0) for d in dec_order]
dec_cols = ['#2ca02c', '#d62728', '#ff7f0e', '#9467bd']
bars2 = ax2.bar(dec_order, dec_vals, color=dec_cols, width=0.55)
ax2.set_title('NBA L2M Review Decisions', fontsize=12, fontweight='bold')
for b in bars2:
    h = b.get_height()
    ax2.text(b.get_x() + b.get_width() / 2, h + 3, f'{h}', ha='center', fontweight='bold')

# 3. Foul Call Taxonomy
top_calls = call_counts.most_common(4)
ax3.barh([c[0] for c in top_calls], [c[1] for c in top_calls], color='#1f77b4', height=0.5)
ax3.set_title('Foul Call Categories', fontsize=12, fontweight='bold')
ax3.set_xlabel('Plays')

plt.tight_layout()
plt.show()


---
## Stage 2: GPU-Accelerated YOLOv11 Biomechanical Pose & Action-Tube Extraction (`2_preprocess_tubes.py`)
Feeding uncompressed 1080p broadcast video clips directly into 3D Convolutional Neural Networks leads to **RAM/VRAM overflow** and **spatial dilution** (interacting players occupy < 8% of the court pixels).

### Our Two-Pass GPU-Accelerated Solution:
1. **Pass 1 (Lightweight Pose Scan)**: Scans the live play window using `yolo11n-pose.pt` directly on CUDA with batch processing (`yolo_batch_size = 16`). It extracts a **12-D Mechanical Foul Feature Vector** grounded in NBA Rule 12 officiating rules:
   - `[0] arm_contact_intrusion`: Defender hand/wrist collision with shooter arm
   - `[1] body_cylinder_penetration`: Torso/hip invasion into offensive cylinder
   - `[2] lower_body_landing_violation`: Defender feet/legs in shooter landing zone
   - `[3] defender_verticality_angle`: Angle deviation from legal 90-degree vertical contest
   - `[4..11]`: Lateral positioning, displacement, ball distances, closing velocity, momentum shock, apex diff.
2. **Pass 2 (Action-Tube Generation)**: Identifies the exact moment of peak physical impact $t^*$, smooths bounding boxes via bidirectional exponential moving averages (EMA), and crops a stabilized `(3, 16, 224, 224)` RGB spatiotemporal action tube.


In [ ]:
# ==============================================================================
# CELL 4: Run Stage 2 GPU-Accelerated Preprocessing
# ==============================================================================
import importlib
preprocess_mod = importlib.import_module('2_preprocess_tubes')
importlib.reload(preprocess_mod)

print("⚙️  Running Stage 2 Biomechanical Preprocessing on GPU...")
stats = preprocess_mod.preprocess_dataset(
    metadata_csv=METADATA_CSV,
    processed_dir=PROCESSED_DIR,
    processed_metadata_csv=PROCESSED_METADATA_CSV,
    test_dir=TEST_DIR,
    yolo_weights=str(WEIGHTS_DIR / 'yolo11n.pt'),
    pose_weights=str(WEIGHTS_DIR / 'yolo11n-pose.pt'),
    window_sec=1.5,
    num_frames=CONFIG['num_frames'],
    crop_size=CONFIG['crop_size'],
    device='cuda' if IS_CUDA else 'cpu',
    max_live_play_sec=CONFIG['max_live_play_sec'],
    max_scan_samples=CONFIG['max_scan_samples'],
    max_clips=CONFIG['max_preprocess_clips'],
    yolo_batch_size=CONFIG['yolo_batch_size'],
    synthetic_fallback=True,
)

processed_files = list(PROCESSED_DIR.glob('*/*.pt'))
print(f"✅ Preprocessing completed! {len(processed_files)} action-tube tensors ready in data/processed/")

import gc
gc.collect()
if IS_CUDA:
    torch.cuda.empty_cache()


---
## Stage 3: Dual-Stream Spatiotemporal Architecture & RL Environment Schema
Our spatiotemporal pipeline fuses two complementary representations into a unified 768-D representation:

1. **Visual Spatiotemporal Stream (`R(2+1)D-18`)**: Decomposes 3D convolutions into $2D$ spatial $+ 1D$ temporal operations over the `(3, 16, 224, 224)` action tube, yielding a **512-D visual embedding**.
2. **Kinematic Temporal Attention Pose Stream (`BiomechanicalFoulEncoder`)**: Projects the `(16, 12)` mechanical foul sequence into a 128-D space, applies **4-Head Temporal Self-Attention** across all 16 frames with residual connection and LayerNorm, followed by dual temporal pooling (mean + max) to produce a **256-D kinematic contact embedding**.
3. **Deep Late Fusion Head**: `768-D -> 512-D -> 256-D -> 1` foul logit (~33.8M parameters).
4. **`NBARefereeEnv`**: Sequential officiating environment ($S_t \in \mathbb{R}^{281}$, $A_t \in \{0, 1\}$) modeling referee decisions across frames.


In [ ]:
# ==============================================================================
# CELL 5: Verify Model Architecture, Trainable Parameters & RL Environment
# ==============================================================================
import importlib
model_mod = importlib.import_module('3_model_and_dataset')
importlib.reload(model_mod)

# Initialize Dual-Stream Model
model = model_mod.FoulClassifier3D(
    pretrained=True,
    dropout_p=CONFIG['dropout'],
    freeze_backbone=CONFIG['freeze_backbone'],
).to(DEVICE)

summary = model.trainable_parameter_summary()
print("🏗️  FoulClassifier3D Architecture Summary:")
print(f"   • Total Parameters    : {summary['total_params']:,}")
print(f"   • Trainable Parameters: {summary['trainable_params']:,} ({'Full End-to-End Fine-Tuning' if not CONFIG['freeze_backbone'] else 'Backbone Frozen'})")
print(f"   • Frozen Parameters   : {summary['frozen_params']:,}")
print(f"   • Video Backbone      : R(2+1)D-18 (512-D Visual Representation)")
print(f"   • Kinematic Encoder   : 4-Head Multi-Head Self-Attention (256-D Pose Representation)")
print(f"   • Combined Dimension  : {model_mod.COMBINED_FEATURE_DIM}-D Fusion Embedding")

# Helper function to extract features across all versions
def extract_model_features(net: nn.Module, video_frames: torch.Tensor, pose_features: torch.Tensor) -> torch.Tensor:
    if hasattr(net, 'extract_features'):
        return net.extract_features(video_frames, pose_features)
    if video_frames.dim() == 2 and video_frames.size(-1) == net.video_feature_dim:
        v_emb = video_frames
    elif getattr(net, 'backbone_frozen', False):
        with torch.no_grad():
            v_emb = net.video_backbone(video_frames)
    else:
        v_emb = net.video_backbone(video_frames)
    p_net = getattr(net, 'pose_network', getattr(net, 'pose_encoder', None))
    p_emb = p_net(pose_features.to(video_frames.device))
    return torch.cat([v_emb, p_emb], dim=-1)

# Reinforcement Learning Dimensions
RL_STATE_DIM = 281
RL_NUM_ACTIONS = 2
ACTION_PLAY_ON = 0
ACTION_WHISTLE = 1
print(f"\n🎮 RL Referee Subsystem Specifications:")
print(f"   • RL State Space : {RL_STATE_DIM}-D (256 visual + 12 contact + 12 delta + 1 clock)")
print(f"   • RL Action Space: 2 Actions (0 = PLAY ON, 1 = WHISTLE FOUL)")
print(f"   • Ground Truth   : NBA L2M League Reviews (CC, INC, CNC, IC) & Rule 12 mechanics")


---
## Stage 4: High-VRAM End-to-End GPU Training (Engaging 12-14 GB of Tesla T4 VRAM)
In this cell, the entire 3D Convolutional Neural Network (`R(2+1)D-18`) is unfreezed and trained end-to-end on raw 4D video tensors `(B, 3, 16, 224, 224)` on the Tesla T4 GPU.

### GPU Compute Utilization Features:
- **Batch Size 16**: Direct feed of 4D video batches into GDDR6 VRAM.
- **Automatic Mixed Precision (AMP FP16)**: `torch.amp.autocast('cuda')` activates Turing Tensor Cores for 3D convolutions.
- **Live VRAM Monitoring**: Reports active and peak GPU memory allocated in Gigabytes.
- **Class-Balanced Focal Loss**: Handles NBA foul distribution with focal scaling (`gamma = 1.5`).
- **Threshold Calibration**: Calibrates optimal decision threshold maximizing F1 and Balanced Accuracy.


In [ ]:
# ==============================================================================
# CELL 6: End-to-End Spatiotemporal GPU Training & Live VRAM Monitoring
# ==============================================================================
import importlib
train_mod = importlib.import_module('4_train_eval')
importlib.reload(train_mod)

train_mod.seed_everything(CONFIG['seed'])

# Setup End-to-End High-VRAM Dataset
base_dataset = model_mod.L2MFoulDataset(
    processed_dir=PROCESSED_DIR,
    metadata_csv=METADATA_CSV,
    is_train=True,
    max_samples=CONFIG.get('max_train_samples', 0),
    return_foul_features=True,
    use_cached_embeddings=CONFIG['use_cached_embeddings'],
)

all_samples = base_dataset.samples
labels = [lbl for _, lbl in all_samples]
stratify_labels = labels if len(set(labels)) > 1 and min(labels.count(0), labels.count(1)) >= 2 else None

train_samples, val_samples = train_test_split(
    all_samples,
    test_size=0.20,
    random_state=CONFIG['seed'],
    stratify=stratify_labels,
)

safe_workers = train_mod.get_safe_num_workers(CONFIG['num_workers'])
use_pin_memory = CONFIG['pin_memory']

train_loader = DataLoader(
    model_mod.L2MFoulDataset(
        samples=train_samples,
        is_train=True,
        return_foul_features=True,
        use_cached_embeddings=CONFIG['use_cached_embeddings'],
    ),
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=safe_workers,
    pin_memory=use_pin_memory,
    collate_fn=train_mod.safe_foul_collate,
)

val_loader = DataLoader(
    model_mod.L2MFoulDataset(
        samples=val_samples,
        is_train=False,
        return_foul_features=True,
        use_cached_embeddings=CONFIG['use_cached_embeddings'],
    ),
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=safe_workers,
    pin_memory=use_pin_memory,
    collate_fn=train_mod.safe_foul_collate,
)

print(f"📊 Training Split: {len(train_samples)} samples | Validation Split: {len(val_samples)} samples")
print(f"⚡ Batch Size: {CONFIG['batch_size']} | Pinned Memory: {use_pin_memory} | Workers: {safe_workers}")

pos_weight = train_mod.compute_pos_weight(train_samples, device=DEVICE)
criterion = train_mod.FoulFocalBCEWithLogitsLoss(pos_weight=pos_weight, gamma=1.5)

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=max(1, CONFIG['epochs']), eta_min=CONFIG['lr'] * 0.01)
scaler = torch.amp.GradScaler('cuda', enabled=CONFIG['amp']) if (IS_CUDA and CONFIG['amp']) else None

best_f1 = -1.0
best_roc_auc = -1.0
history = {'train_loss': [], 'val_loss': [], 'f1': [], 'acc': [], 'roc_auc': []}

print("\n🚀 Beginning Spatiotemporal Training Loop...")
start_time = time.time()

for epoch in range(1, CONFIG['epochs'] + 1):
    train_loss = train_mod.train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=DEVICE,
        accum_steps=CONFIG['accum_steps'],
    )
    val_loss, metrics = train_mod.evaluate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE,
    )
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['f1'].append(metrics['f1'])
    history['acc'].append(metrics['accuracy'])
    history['roc_auc'].append(metrics['roc_auc'])

    vram_str = ""
    if IS_CUDA:
        vram_alloc = torch.cuda.memory_allocated() / (1024**3)
        vram_max = torch.cuda.max_memory_allocated() / (1024**3)
        vram_str = f" | VRAM: {vram_alloc:.1f}/{vram_max:.1f} GB"

    print(f"Epoch [{epoch:02d}/{CONFIG['epochs']:02d}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Acc: {metrics['accuracy']:.3f} | F1: {metrics['f1']:.3f} | ROC-AUC: {metrics['roc_auc']:.3f}{vram_str}")

    if metrics['f1'] > best_f1 or (np.isclose(metrics['f1'], best_f1) and metrics['roc_auc'] > best_roc_auc):
        best_f1 = metrics['f1']
        best_roc_auc = metrics['roc_auc']
        CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics': metrics,
            'optimal_threshold': metrics['threshold'],
            'val_loss': val_loss,
        }, CHECKPOINT_PATH)

total_elapsed = time.time() - start_time
print(f"\n🏁 Training Completed in {total_elapsed:.1f}s! Best Validation F1: {best_f1:.3f} | Best ROC-AUC: {best_roc_auc:.3f}")
print(f"💾 Best checkpoint saved to: {CHECKPOINT_PATH}")

# Plot Training Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), dpi=100)
epochs_range = range(1, len(history['train_loss']) + 1)
ax1.plot(epochs_range, history['train_loss'], 'o-', label='Train Loss', color='#1f77b4')
ax1.plot(epochs_range, history['val_loss'], 's--', label='Val Loss', color='#ff7f0e')
ax1.set_title('Spatiotemporal Focal Loss Convergence', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(epochs_range, history['f1'], 'o-', label='F1 Score', color='#2ca02c')
ax2.plot(epochs_range, history['acc'], 's--', label='Accuracy', color='#9467bd')
ax2.plot(epochs_range, history['roc_auc'], '^:', label='ROC-AUC', color='#d62728')
ax2.set_title('Validation Metrics & Discrimination', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score')
ax2.grid(True, alpha=0.3)
ax2.legend()
plt.tight_layout()
plt.show()


---
## Stage 5: Reinforcement Learning (RL) Referee Officiating Agent
In actual NBA basketball, refereeing is a **sequential decision process under uncertainty**:
- Legal physical contact is constantly permitted (fighting through screens, legal guarding position, vertical rim protection).
- Contact only constitutes an infraction if an offensive player is illegally disadvantaged (cylinder breach, arm hack, verticality deviation, landing space violation).
- The referee agent must decide at every frame:
  - `Action 0 (PLAY_ON)`: Hold the whistle; allow physical legal play to continue.
  - `Action 1 (WHISTLE_FOUL)`: Blow the whistle; stop play and assess a foul.

### Reward Function Grounded in NBA Official L2M Reviews:
- **Whistle on `CC` (Correct Call)**: **`+2.5`** + timing precision bonus.
- **Whistle on `INC` (League Overturn)**: **`+3.0`** (Rewards agent for detecting a foul missed by real floor referees!).
- **Play on `CNC` (Correct Non-Call)**: **`+2.0`** (Rewards agent for discipline during aggressive legal defense).
- **Play on `IC` (League Overturn)**: **`+2.5`** (Rewards agent for avoiding a phantom whistle blown by real refs!).
- **Whistle on `CNC` / `IC` (Phantom Whistle)**: **`-3.0`** (Penalizes interrupting clean basketball defense).
- **Play on `CC` / `INC` (Swallowed Whistle)**: **`-2.5`** (Penalizes failure to protect the shooter).


In [ ]:
# ==============================================================================
# CELL 7: Train Reinforcement Learning (RL) Referee Officiating Agent
# ==============================================================================

# ------------------------------------------------------------------------------
# Self-Contained Gym-Like Officiating Environment & Dueling Q-Network
# ------------------------------------------------------------------------------
class NBARefereeEnvNotebook:
    def __init__(self, episodes: List[Dict[str, Any]]) -> None:
        self.episodes = episodes
        self.current_idx = 0
        self.current_t = 0
        self.current_episode: Optional[Dict[str, Any]] = None

    def _get_state(self, t: int) -> np.ndarray:
        ep = self.current_episode
        T = ep["num_frames"]
        v_feat = ep["visual_sequence"][min(t, T - 1)][:256]
        b_feat = ep["pose_features"][min(t, T - 1)]
        delta_b = b_feat - ep["pose_features"][t - 1] if t > 0 else np.zeros_like(b_feat)
        clock = np.array([float(t) / max(1.0, float(T - 1))], dtype=np.float32)
        return np.concatenate([v_feat, b_feat, delta_b, clock]).astype(np.float32)

    def reset(self, idx: Optional[int] = None) -> np.ndarray:
        self.current_idx = (idx if idx is not None else random.randint(0, len(self.episodes) - 1)) % len(self.episodes)
        self.current_episode = self.episodes[self.current_idx]
        self.current_t = 0
        return self._get_state(0)

    def step(self, action: int) -> Tuple[np.ndarray, float, bool, Dict[str, Any]]:
        ep = self.current_episode
        t = self.current_t
        T = ep["num_frames"]
        label = ep["label"]
        decision = ep.get("review_decision", "CC" if label == 1 else "CNC")
        peak_f = ep.get("peak_frame", T // 2)
        pose_t = ep["pose_features"][min(t, T - 1)]
        cyl = float(pose_t[1]) if len(pose_t) > 1 else 0.0
        vert = float(pose_t[3]) if len(pose_t) > 3 else 0.0

        done = False
        reward = 0.0
        info = {"action": action, "true_label": label, "frame": t, "correct_call": False}

        if action == ACTION_WHISTLE:
            done = True
            info["whistle_blown"] = True
            if label == 1:
                timing = 1.0 * max(0.0, 1.0 - abs(t - peak_f) / 4.0)
                reward = 2.5 + timing + (0.5 if decision == "INC" else 0.0)
                info["correct_call"] = True
            else:
                reward = -3.0 - (0.5 if vert < 0.20 else 0.0)
        else:
            if t >= T - 1:
                done = True
                info["whistle_blown"] = False
                if label == 0:
                    reward = 2.0 + (0.5 if decision == "IC" else 0.0)
                    info["correct_call"] = True
                else:
                    reward = -2.5
            else:
                done = False
                self.current_t += 1
                if vert < 0.25 and cyl < 0.35:
                    reward = 0.05
                elif cyl > 0.65:
                    reward = -0.10

        next_state = self._get_state(self.current_t) if not done else np.zeros((RL_STATE_DIM,), dtype=np.float32)
        return next_state, reward, done, info

class RefereeDQNNotebook(nn.Module):
    def __init__(self, state_dim: int = 281, num_actions: int = 2) -> None:
        super().__init__()
        self.feat_net = nn.Sequential(
            nn.Linear(state_dim, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Linear(256, 128), nn.LayerNorm(128), nn.GELU(),
        )
        self.v_head = nn.Sequential(nn.Linear(128, 64), nn.GELU(), nn.Linear(64, 1))
        self.a_head = nn.Sequential(nn.Linear(128, 64), nn.GELU(), nn.Linear(64, num_actions))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        f = self.feat_net(x)
        v = self.v_head(f)
        a = self.a_head(f)
        return v + (a - a.mean(dim=-1, keepdim=True))

class RefereeRLAgentNotebook:
    def __init__(self, device: torch.device) -> None:
        self.device = device
        self.model = RefereeDQNNotebook().to(device)
        self.target = RefereeDQNNotebook().to(device)
        self.target.load_state_dict(self.model.state_dict())
        self.optimizer = AdamW(self.model.parameters(), lr=CONFIG['rl_lr'], weight_decay=1e-3)
        self.buffer = deque(maxlen=20000)
        self.eps = 1.0

    def select_action(self, st: np.ndarray, evaluate: bool = False) -> int:
        if not evaluate and random.random() < self.eps:
            return random.randint(0, 1)
        t_st = torch.from_numpy(st).float().unsqueeze(0).to(self.device)
        self.model.eval()
        with torch.no_grad():
            return int(torch.argmax(self.model(t_st), dim=-1).item())

    def store(self, s, a, r, s_next, d):
        self.buffer.append((s, a, r, s_next, d))

    def train_step(self, batch_size: int = 64) -> float:
        if len(self.buffer) < batch_size:
            return 0.0
        b = random.sample(self.buffer, batch_size)
        s = torch.from_numpy(np.array([x[0] for x in b], dtype=np.float32)).to(self.device)
        a = torch.tensor([x[1] for x in b], dtype=torch.long, device=self.device)
        r = torch.tensor([x[2] for x in b], dtype=torch.float32, device=self.device)
        s_next = torch.from_numpy(np.array([x[3] for x in b], dtype=np.float32)).to(self.device)
        d = torch.tensor([x[4] for x in b], dtype=torch.float32, device=self.device)

        self.model.train()
        self.optimizer.zero_grad()
        curr_q = self.model(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q = self.target(s_next).max(dim=1).values
            target_q = r + (1.0 - d) * 0.95 * next_q
        loss = F.smooth_l1_loss(curr_q, target_q)
        loss.backward()
        self.optimizer.step()
        if self.eps > 0.05:
            self.eps *= 0.995
        return float(loss.item())

    def evaluate_trajectory(self, v_seq: np.ndarray, p_seq: np.ndarray) -> Dict[str, Any]:
        T = len(p_seq)
        if v_seq.ndim == 1:
            v_seq = np.tile(v_seq[:256], (T, 1))
        elif len(v_seq) != T:
            idxs = np.linspace(0, len(v_seq) - 1, T).round().astype(int)
            v_seq = v_seq[idxs, :256]
        else:
            v_seq = v_seq[:, :256]

        q_hist = []
        p_hist = []
        whistle_f = None
        self.model.eval()
        with torch.no_grad():
            for t in range(T):
                vt = v_seq[t]
                bt = p_seq[t]
                db = bt - p_seq[t - 1] if t > 0 else np.zeros_like(bt)
                clk = np.array([float(t) / max(1.0, float(T - 1))], dtype=np.float32)
                st = torch.from_numpy(np.concatenate([vt, bt, db, clk]).astype(np.float32)).unsqueeze(0).to(self.device)
                q_vals = self.model(st)[0].cpu().numpy()
                q_hist.append([float(q_vals[0]), float(q_vals[1])])
                exp_q = np.exp(q_vals - np.max(q_vals))
                p_w = float((exp_q / exp_q.sum())[1])
                p_hist.append(p_w)
                if whistle_f is None and q_vals[1] > q_vals[0]:
                    whistle_f = t
        is_f = (whistle_f is not None)
        return {
            "decision": "FOUL_WHISTLED" if is_f else "LEGAL_PLAY_ON",
            "whistle_frame": whistle_f,
            "confidence": float(p_hist[whistle_f]) if is_f else float(1.0 - max(p_hist)),
            "whistle_probs": p_hist,
            "q_values": q_hist,
        }

# Extract episodes from preprocessed dataset
print("🎮 Formatting dataset into sequential game episodes for NBARefereeEnv...")
episodes = []
for p, lbl in base_dataset.samples:
    raw_v, p_feats, v_emb = base_dataset._load_video_and_features(p)
    if v_emb is None:
        v_emb = base_dataset._extract_and_cache_embedding(raw_v, p)
    p_mat = p_feats.cpu().numpy() if isinstance(p_feats, torch.Tensor) else np.array(p_feats)
    v_vec = v_emb.cpu().numpy()[:256]
    c_scores = p_mat[:, 0] * 0.5 + p_mat[:, 1] * 0.5
    peak_f = int(np.argmax(c_scores)) if len(c_scores) > 0 else 8
    episodes.append({
        "play_id": p.stem,
        "num_frames": len(p_mat),
        "visual_sequence": np.tile(v_vec, (len(p_mat), 1)),
        "pose_features": p_mat,
        "label": int(lbl),
        "review_decision": "CC" if lbl == 1 else "CNC",
        "peak_frame": peak_f,
    })

print(f"📋 Created {len(episodes)} trajectory episodes ({sum(1 for e in episodes if e['label'] == 1)} fouls, {sum(1 for e in episodes if e['label'] == 0)} legal plays)")

# Train RL Agent
rl_env = NBARefereeEnvNotebook(episodes)
rl_agent = RefereeRLAgentNotebook(DEVICE)
rl_rewards, rl_correct, rl_losses = [], [], []

print(f"🤖 Training Referee RL Agent (Dueling DQN) across {CONFIG['rl_episodes']} episodes...")
for ep_i in range(1, CONFIG['rl_episodes'] + 1):
    s = rl_env.reset()
    d = False
    ep_r = 0.0
    while not d:
        a = rl_agent.select_action(s)
        s_next, r, d, info = rl_env.step(a)
        rl_agent.store(s, a, r, s_next, d)
        l = rl_agent.train_step(batch_size=64)
        if l > 0:
            rl_losses.append(l)
        s = s_next
        ep_r += r
    if ep_i % 5 == 0:
        rl_agent.target.load_state_dict(rl_agent.model.state_dict())
    rl_rewards.append(ep_r)
    rl_correct.append(bool(info.get("correct_call", False)))
    if ep_i % 25 == 0 or ep_i == CONFIG['rl_episodes']:
        rec_acc = float(np.mean(rl_correct[-25:]) * 100.0)
        rec_rew = float(np.mean(rl_rewards[-25:]))
        print(f"RL Episode [{ep_i:03d}/{CONFIG['rl_episodes']:03d}] | Avg Return: {rec_rew:+.2f} | Accuracy: {rec_acc:.1f}% | Epsilon: {rl_agent.eps:.2f}")

# Save trained weights
RL_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save({"model_state_dict": rl_agent.model.state_dict(), "epsilon": rl_agent.eps}, str(RL_CHECKPOINT_PATH))
print(f"\n💾 Saved trained Referee RL Agent checkpoint to: {RL_CHECKPOINT_PATH}")

# Plot RL Learning Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), dpi=100)
ep_x = range(1, len(rl_rewards) + 1)
ax1.plot(ep_x, rl_rewards, color='#1f77b4', alpha=0.35, label='Episode Return')
window = 15
if len(rl_rewards) >= window:
    rolling = np.convolve(rl_rewards, np.ones(window)/window, mode='valid')
    ax1.plot(range(window, len(rl_rewards) + 1), rolling, color='#d62728', lw=2, label=f'{window}-Ep Rolling Avg')
ax1.set_title('RL Referee Episode Return (L2M Reward)', fontweight='bold')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Return')
ax1.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax1.grid(True, alpha=0.3)
ax1.legend()

if rl_losses:
    ax2.plot(range(1, len(rl_losses) + 1), rl_losses, color='#2ca02c')
    ax2.set_title('Dueling DQN Bellman Loss Convergence', fontweight='bold')
    ax2.set_xlabel('Gradient Step')
    ax2.set_ylabel('Huber Loss')
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## Stage 6: RL Whistle Policy Telemetry & Sequential Decision Telemetry
The cell below evaluates the trained `RefereeRLAgent` on an unseen play from the validation split. It traces the frame-by-frame evolution of:
1. **Dynamic Q-Values**: $Q(\text{Whistle})$ vs. $Q(\text{Play On})$ across the 16-frame contest window.
2. **Biomechanical Violation Surges**: Cylinder penetration and arm contact intrusion, illustrating the exact moment where $Q(\text{Whistle}) > Q(\text{Play On})$ triggers the whistle.


In [ ]:
# ==============================================================================
# CELL 8: Frame-by-Frame RL Decision Telemetry & Whistle Evolution
# ==============================================================================
eval_sample_idx = 0
sample_path, sample_lbl = val_samples[eval_sample_idx]
raw_vid, sample_pose, sample_emb = base_dataset._load_video_and_features(sample_path)
if sample_emb is None:
    sample_emb = base_dataset._extract_and_cache_embedding(raw_vid, sample_path)

v_np = sample_emb.cpu().numpy() if isinstance(sample_emb, torch.Tensor) else np.array(sample_emb)
p_np = sample_pose.cpu().numpy() if isinstance(sample_pose, torch.Tensor) else np.array(sample_pose)

rl_results = rl_agent.evaluate_trajectory(v_np, p_np)

print(f"🎬 Play Evaluated: {sample_path.stem}")
print(f"   • Ground Truth Label : {'FOUL' if sample_lbl == 1 else 'NO FOUL'}")
print(f"   • RL Referee Decision: {rl_results['decision']} (Confidence: {rl_results['confidence'] * 100:.1f}%)")
print(f"   • Whistle Frame      : {rl_results['whistle_frame']} / {CONFIG['num_frames']}")

frames_x = range(1, CONFIG['num_frames'] + 1)
q_play = [q[0] for q in rl_results['q_values']]
q_foul = [q[1] for q in rl_results['q_values']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5), dpi=100)

ax1.plot(frames_x, q_play, 'o-', color='#2ca02c', label='Q(Play On / Clean)', lw=2)
ax1.plot(frames_x, q_foul, 's-', color='#d62728', label='Q(Whistle Foul)', lw=2)
if rl_results['whistle_frame'] is not None:
    wf = rl_results['whistle_frame'] + 1
    ax1.axvline(wf, color='red', linestyle='--', lw=2, label=f'Whistle Blown (Frame {wf})')
ax1.set_title(f'Sequential Action-Value Trajectory — {sample_path.stem}', fontweight='bold')
ax1.set_xlabel('Frame Index (16-Frame Window)')
ax1.set_ylabel('Action-Value Q(s, a)')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(frames_x, p_np[:, 0], 'o-', label='Arm Contact Intrusion', color='#e377c2')
ax2.plot(frames_x, p_np[:, 1], 's-', label='Cylinder Penetration', color='#ff7f0e')
ax2.plot(frames_x, p_np[:, 3], '^-', label='Verticality Angle Deviation', color='#17becf')
ax2.plot(frames_x, p_np[:, 11], 'd--', label='Apex Contest Intensity', color='#bcbd22')
ax2.set_title('Rule 12 Biomechanical Contact Profile', fontweight='bold')
ax2.set_xlabel('Frame Index')
ax2.set_ylabel('Normalized Severity [0..1]')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()


---
## Stage 7: Continuous Multimodal Video Annotation & Test Inference
Our inference engine processes broadcast video clips into referee-grade annotated MP4s saved to `data/annotated/<clip_id>_annotated.mp4`.

### Visual Annotation Features:
1. **Multi-Player Identity-Preserved Tracking**: Enclosing hull around both the offensive ball handler/shooter (Cyan) and contesting defender (Orange) across 100% of video frames.
2. **Defender Verticality Guideline**: Displays legal 90-degree normal dashed line vs. defender contest angle with real-time status (`[LEGAL]` vs `[BREACH]`).
3. **Top-Left HUD Panel**: Displays officiating status, foul taxonomy, player matchup, live RL Q-values ($Q(\text{Foul})$ vs $Q(\text{Play On})$), whistle probabilities, and Rule 12 diagnostic bars.
4. **Bottom Temporal Scrubbing Bar**: Progress timeline with green (legal play) and red (whistle blown) segments plus scrubbing playhead pip.


In [ ]:
# ==============================================================================
# CELL 9: Continuous Video Annotation on Dedicated Test Clips (data/test/)
# ==============================================================================
import importlib
train_mod = importlib.import_module("4_train_eval")
importlib.reload(train_mod)

test_clips = list(TEST_DIR.glob("*.mp4")) if TEST_DIR.exists() else []

# Fallback to downloaded raw clips if data/test/ has no MP4s in fresh clone
if not test_clips and RAW_DIR.exists():
    raw_cands = list(RAW_DIR.glob("*/*.mp4"))
    if raw_cands:
        print(f"ℹ️  data/test/ is empty; falling back to downloaded raw clip: {raw_cands[0].name}")
        test_clips = [raw_cands[0]]

print(f"🎬 Found {len(test_clips)} clips for continuous video annotation: {[c.name for c in test_clips]}")

if test_clips:
    sample_test_clip = test_clips[0]
    output_mp4 = ANNOTATED_DIR / f"{sample_test_clip.stem}_annotated.mp4"
    print(f"🚀 Running Continuous Video Annotation on {sample_test_clip.name}...")
    
    p_foul = train_mod.predict_single_clip(
        video_path=sample_test_clip,
        checkpoint_path=CHECKPOINT_PATH,
        output_mp4_path=output_mp4,
        yolo_weights=str(WEIGHTS_DIR / "yolo11n.pt"),
        pose_weights=str(WEIGHTS_DIR / "yolo11n-pose.pt"),
    )
    print("")
    print(f"✅ Successfully generated annotated MP4 video: {output_mp4}")
    print(f"   • P(Foul): {p_foul * 100:.1f}%")
else:
    print("⚠️ No test clips found in data/test/ or data/raw/. Run Cell 2 to scrape clips first.")


In [ ]:
# ==============================================================================
# CELL 10: In-Notebook Interactive HTML5 Video Player & Telemetry Inspector
# ==============================================================================
from IPython.display import HTML, display
import base64

annotated_videos = list(ANNOTATED_DIR.glob('*_annotated.mp4')) if ANNOTATED_DIR.exists() else []

if annotated_videos:
    display_vid = annotated_videos[0]
    print(f"📺 Displaying annotated replay: {display_vid.name}")
    
    with open(display_vid, 'rb') as f:
        video_bytes = f.read()
    b64_vid = base64.b64encode(video_bytes).decode('utf-8')
    
    html_code = f'''
    <div style="background-color: #1a1a1a; padding: 15px; border-radius: 8px; border: 2px solid #00c853; max-width: 900px;">
        <h3 style="color: #ffffff; margin-top: 0;">🏀 NBA Referee AI — Continuous Annotated Replay</h3>
        <p style="color: #cccccc; font-size: 13px;">
            Tracking: <span style="color: #00e5ff; font-weight: bold;">Cyan Box = Offense</span> | 
            <span style="color: #ff9100; font-weight: bold;">Orange Box = Defense</span> | 
            <span style="color: #00e676; font-weight: bold;">Green Hull = Legal Play</span> ➔ 
            <span style="color: #ff1744; font-weight: bold;">Red Hull = Whistle Blown</span>
        </p>
        <video width="100%" controls autoplay loop muted style="border-radius: 4px;">
            <source src="data:video/mp4;base64,{b64_vid}" type="video/mp4">
            Your browser does not support HTML5 video.
        </video>
    </div>
    '''
    display(HTML(html_code))
else:
    print("⚠️ No annotated videos found in data/annotated/. Run Cell 9 first.")
